# Task 4 ensemble: which of two books did the user read?

Validation pairs are built like the test pairs: a held-out interaction against the book of a random other interaction
(drawn in proportion to popularity). Sub-models score (user, book); a LightGBM pair blender decides every pair.
The pair blender also sees each book's popularity around the timestamp and its age at that time (the test negatives are drawn regardless of time).

- **popularity**: log(1 + #interactions of the book)
- **item cosine**: similarity of the book to the books the user has read
- **EASE**: linear autoencoder in closed form (Steck 2019)
- **time-weighted EASE**: the same weights, the user's books read close to the asked day count more
- **Mult-VAE**: non-linear variational autoencoder (Liang et al. 2018), trained on the GPU with early stopping

Every sub-model cell stores its scores in `trained_models/task4/<model>.pt`; the blend cells only need the stored files.
All hyperparameters: `submission/lib/config.py`, Task 4 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live in ./submission (this notebook lives in assignment1_2026)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "submission"))

import torch

import assignment1
from lib.config import TASK4_PAIR_BLENDER
from lib.models.ease_model import load_ease_model, load_time_ease_model, store_ease_model, store_time_ease_model
from lib.models.item_cosine_model import load_item_cosine_model, store_item_cosine_model
from lib.models.mult_vae import load_mult_vae, store_mult_vae
from lib.models.pair_blender import load_pair_blender, store_pair_blender
from lib.models.popularity_model import load_popularity_model, store_popularity_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


In [2]:
# training file -> validation pairs (held-out interactions vs. popularity-drawn negatives) + the test pairs
task4 = assignment1.load_task4()

## Sub-models

In [3]:
# popularity: log(1 + #interactions of the book) (seconds)
popularity_result = assignment1.train_task4_popularity_model(task4)
store_popularity_model(popularity_result, "task4")

[01:39:56] popularity | validation accuracy 0.4975
[01:39:56] stored popularity -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task4\popularity.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task4/popularity.pt')

In [4]:
# item cosine similarity to the user's history (CPU, about a minute)
item_cosine_result = assignment1.train_task4_item_cosine_model(task4)
store_item_cosine_model(item_cosine_result, "task4")

[01:40:03] item cosine | validation accuracy 0.6769
[01:40:10] stored item_cosine -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task4\item_cosine.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task4/item_cosine.pt')

In [5]:
# EASE: one Gram matrix + one matrix inverse per fit (GPU, seconds); only the scores are stored, not the dense weights
ease_result = assignment1.train_task4_ease_model(task4)
store_ease_model(ease_result, "task4")

[01:40:13] EASE | validation accuracy 0.7514
[01:40:15] stored ease -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task4\ease.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task4/ease.pt')

In [6]:
# EASE with a time-weighted history: books read close to the asked day count more (GPU, under a minute)
time_ease_result = assignment1.train_task4_time_ease_model(task4)
store_time_ease_model(time_ease_result, "task4")

[01:40:17] time-weighted EASE | validation accuracy 0.7872
[01:40:20] stored time_ease -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task4\time_ease.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task4/time_ease.pt')

In [7]:
# Mult-VAE: non-linear autoencoder, early stopping on the validation pair accuracy, then refit on all interactions (GPU)
mult_vae_result = assignment1.train_task4_mult_vae(task4)
store_mult_vae(mult_vae_result, "task4")

[01:40:24] Mult-VAE epoch   1 | train loss 293.73 | validation accuracy 0.5088
[01:40:26] Mult-VAE epoch   2 | train loss 279.00 | validation accuracy 0.6142
[01:40:28] Mult-VAE epoch   3 | train loss 268.61 | validation accuracy 0.6607
[01:40:30] Mult-VAE epoch   4 | train loss 264.66 | validation accuracy 0.6836
[01:40:32] Mult-VAE epoch   5 | train loss 261.60 | validation accuracy 0.6939
[01:40:34] Mult-VAE epoch   6 | train loss 259.19 | validation accuracy 0.7072
[01:40:36] Mult-VAE epoch   7 | train loss 256.22 | validation accuracy 0.7176
[01:40:38] Mult-VAE epoch   8 | train loss 254.40 | validation accuracy 0.7218
[01:40:40] Mult-VAE epoch   9 | train loss 252.54 | validation accuracy 0.7252
[01:40:42] Mult-VAE epoch  10 | train loss 251.16 | validation accuracy 0.7319
[01:40:44] Mult-VAE epoch  11 | train loss 249.81 | validation accuracy 0.7352
[01:40:47] Mult-VAE epoch  12 | train loss 248.51 | validation accuracy 0.7383
[01:40:49] Mult-VAE epoch  13 | train loss 247.37 | 

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task4/mult_vae.pt')

## Blend

In [8]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "popularity": load_popularity_model,
    "item_cosine": load_item_cosine_model,
    "ease": load_ease_model,
    "time_ease": load_time_ease_model,
    "mult_vae": load_mult_vae,
}
task4_model_results = {model_name: sub_model_loaders[model_name]("task4") for model_name in TASK4_PAIR_BLENDER.combined_models}

In [9]:
# LightGBM pair blender: P(item1 is the real one) from both books' scores + meta-features (prints the validation accuracies)
pair_blend_result = assignment1.train_task4_blender(task4, task4_model_results)
store_pair_blender(pair_blend_result, "task4")

[01:42:09] popularity             validation accuracy 0.4975
[01:42:09] item_cosine            validation accuracy 0.6769
[01:42:09] ease                   validation accuracy 0.7514
[01:42:09] time_ease              validation accuracy 0.7872
[01:42:09] mult_vae               validation accuracy 0.7476
[01:42:09] pair blend             validation accuracy 0.9319  (out-of-fold, 5 folds)
[01:42:11] pair blend inputs: popularity_difference 1326, mult_vae_difference 1150, log_interactions_around_t_difference 978, time_ease_difference 822, item_age_years_at_t_first 697, item_age_years_at_t_second 681, popularity_first 549, time_ease_first 538, popularity_second 534, time_ease_second 519, log_user_interactions 390, ease_second 363, item_age_years_at_t_difference 358, log_interactions_around_t_second 341, ease_difference 337, ease_first 327, log_interactions_around_t_first 327, mult_vae_second 327, item_cosine_second 325, already_read_difference 315, mult_vae_first 311, item_cosine_first 295

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task4/pair_blender.pt')

In [10]:
# writes submission/output/task_4_predictions.csv (the "which" column = the chosen book)
submitted_blend_result = load_pair_blender("task4")
assignment1.write_task4_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK4_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_4_predictions.csv
